# Notebook 01m — PPO baseline on `MountainCar-v0`

Train the standard PPO baseline for MountainCar using `config/envs/mountaincar.yaml`.
This is the original pipeline sanity-check environment: PPO should learn, but it
needs the project-specific settings in the YAML rather than a generic PPO recipe.

The important details are:

- training uses `max_episode_steps=500` so the first goal reach is discoverable;
- greedy evaluation below uses the standard 200-step limit;
- entropy is adaptive because uniform random exploration never reaches the goal;
- actor and critic gradients are clipped separately because the value loss is
  large from the first update.


---
## Knobs


In [ ]:
# ----------------------------------------------------------------------------
# EDIT ME
# ----------------------------------------------------------------------------
ENV_CONFIG    = "mountaincar"      # config/envs/mountaincar.yaml
FORCE_RETRAIN = True

OVERRIDES = {
    # "ppo.total_timesteps": 100_000,   # quick smoke run
    # "run.seeds": (0,),                # one seed while iterating
    # "run.run_name": "mountaincar_smoke",
}
# ----------------------------------------------------------------------------


## 0. Setup


In [ ]:
%load_ext autoreload
%autoreload 2

import sys, pathlib, time

ROOT = pathlib.Path.cwd()
if not (ROOT / "config").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gymnasium as gym

from config import make_config, seed_dir, RUNS_DIR, FIGURES_DIR
from dataio import load_trajectories, validate, load_checkpoint, list_checkpoints
from utils.logging import read_scalars
from utils.plotting import (
    plot_learning_curves, plot_episode_returns, plot_grid,
    plot_state_visitation, savefig,
)
from scripts.train import run_seeds

cfg = make_config(ENV_CONFIG, **OVERRIDES)
RUN_NAME = cfg.run.run_name
SEEDS = tuple(cfg.run.seeds)
FIG = FIGURES_DIR / f"nb01_{RUN_NAME}"
FIG.mkdir(parents=True, exist_ok=True)

print("project root:", ROOT)
print(cfg.summary())


## 1. Train

`run_seeds` writes `scalars.csv`, `episodes.csv`, `trajectories.npz`, checkpoints,
and the exact resolved config under `runs/<run_name>/`.


In [ ]:
already = all((seed_dir(RUN_NAME, s) / "scalars.csv").exists() for s in SEEDS)

if already and not FORCE_RETRAIN:
    print(f"found existing run at {RUNS_DIR / RUN_NAME} -- skipping training")
    print("set FORCE_RETRAIN = True to re-run")
    results = None
else:
    t0 = time.time()
    results = run_seeds(cfg)
    print(f"total wall time: {(time.time() - t0) / 60:.1f} min")


## 2. Load Outputs


In [ ]:
scalars  = {s: read_scalars(seed_dir(RUN_NAME, s) / "scalars.csv") for s in SEEDS}
episodes = {s: pd.read_csv(seed_dir(RUN_NAME, s) / "episodes.csv") for s in SEEDS}
trajs    = ({s: load_trajectories(seed_dir(RUN_NAME, s) / "trajectories.npz") for s in SEEDS}
            if cfg.run.record_trajectories else {})

summary = pd.DataFrame({
    "first_goal_reach": {s: (int(e.loc[e["success"], "global_step"].iloc[0])
                             if e["success"].any() else None) for s, e in episodes.items()},
    "final_return":     {s: d["mean_return_100"].iloc[-1] for s, d in scalars.items()},
    "final_success":    {s: d["success_rate_100"].iloc[-1] for s, d in scalars.items()},
    "final_entropy":    {s: d["entropy"].iloc[-1] for s, d in scalars.items()},
    "final_expl_var":   {s: d["explained_variance"].iloc[-1] for s, d in scalars.items()},
    "episodes":         {s: len(e) for s, e in episodes.items()},
})
summary.index.name = "seed"
display(summary.round(3))


## 3. Return And Success


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
plot_episode_returns(episodes, window=50, ax=axes[0])
plot_learning_curves(scalars, y="success_rate_100",
                     ylabel="goal-reach rate (last 100 ep)",
                     title="Success rate", ax=axes[1])
axes[1].set_ylim(-0.05, 1.05)
fig.tight_layout()
savefig(fig, FIG / "return_curve.png")
plt.show()


## 4. Optimisation Diagnostics


In [ ]:
keys = [k for k in [
    "entropy", "entropy_target", "ent_coef", "approx_kl", "clipfrac",
    "adv_std_raw", "explained_variance", "pg_loss", "v_loss",
    "grad_norm_actor", "grad_norm_critic",
] if any(k in d.columns for d in scalars.values())]
fig = plot_grid(scalars, keys, ncols=3, figsize=(14, 10))
savefig(fig, FIG / "diagnostics.png")
plt.show()


## 5. State Visitation


In [ ]:
if trajs:
    s0 = SEEDS[0]
    fig, ax = plt.subplots(figsize=(5.5, 4))
    plot_state_visitation(trajs[s0].raw_obs, ax=ax)
    fig.suptitle(f"seed {s0} -- position/velocity visitation", y=1.02, fontsize=11)
    savefig(fig, FIG / "state_visitation.png")
    plt.show()
else:
    print("no trajectories recorded")


## 6. Checkpoints And Greedy Evaluation

Training uses a 500-step episode limit, but the policy should also be evaluated
under the standard 200-step MountainCar limit.


In [ ]:
def greedy_eval(ck, n_episodes=50, limit=200, seed0=90_000):
    env = gym.make("MountainCar-v0", max_episode_steps=limit)
    rets, succ = [], 0
    for ep in range(n_episodes):
        obs, _ = env.reset(seed=seed0 + ep)
        total = 0.0
        while True:
            a = int(np.argmax(ck.probs(np.asarray(obs)[None, :])[0]))
            obs, r, term, trunc, _ = env.step(a)
            total += r
            if term:
                succ += 1
                break
            if trunc:
                break
        rets.append(total)
    env.close()
    return float(np.mean(rets)), succ / n_episodes

rows = []
for s in SEEDS:
    for p in list_checkpoints(seed_dir(RUN_NAME, s) / "checkpoints"):
        ck = load_checkpoint(p)
        ret, sr = greedy_eval(ck)
        rows.append({"seed": s, "frac": f"{ck.fraction:.0%}", "step": ck.global_step,
                     "greedy_return_200": round(ret, 1), "greedy_success_200": sr})

FRAC_ORDER = [f"{f:.0%}" for f in cfg.run.checkpoint_fractions]
evaldf = pd.DataFrame(rows)
evaldf["frac"] = pd.Categorical(evaldf["frac"], categories=FRAC_ORDER, ordered=True)
display(evaldf.pivot(index="seed", columns="frac", values=["greedy_return_200", "greedy_success_200"]))
evaldf


## 7. Trajectory Validation


In [ ]:
all_clean = True
for s, t in trajs.items():
    p = seed_dir(RUN_NAME, s) / "trajectories.npz"
    print(f"seed {s}  ({p.stat().st_size / 1e6:.1f} MB)")
    problems = validate(t, n_actions=t.n_actions)
    print("  ->", problems if problems else "NO PROBLEMS")
    all_clean &= not problems
    print()
print("TRAJECTORY VALIDATION:", "PASS" if all_clean else "FAIL")


## 8. Gate 1 Verdict


In [ ]:
best_greedy = {s: evaldf[evaldf.seed == s]["greedy_success_200"].max() for s in SEEDS}
checks = {
    "every seed reaches the goal during training":
        all(episodes[s]["success"].any() for s in SEEDS),
    "every seed reaches 100% greedy success at 200-step eval at some checkpoint":
        all(best_greedy[s] >= 1.0 for s in SEEDS),
    "critic is informative (median explained variance > 0.3)":
        all(scalars[s]["explained_variance"].median() > 0.3 for s in SEEDS),
    "the rollouts carried signal (median adv_std_raw > 1e-4)":
        all(scalars[s]["adv_std_raw"].median() > 1e-4 for s in SEEDS),
    "trajectory files reload and validate": all_clean,
}
width = max(len(k) for k in checks)
for k, v in checks.items():
    print(f"  {'PASS' if v else 'FAIL'}  {k:<{width}}")
print()
print("GATE 1:", "PASS -- baseline learns and the dataset is sound"
      if all(checks.values()) else "FAIL -- inspect diagnostics before PPO-CF")
